# 01 - Exploratory Analysis

**Section 2 of the assignment: characterizing the Internet-traffic signal
before modeling it.**

Every plot and statistic below is produced through `forecasting.SquareSeries`
- the same class `02_experiments.ipynb` uses to load training data - so the
series analyzed here is *exactly* what the models will later be fit and
evaluated on, not a separate ad-hoc read.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from forecasting.data import SquareSeries
from forecasting.viz import apply_style, CATEGORICAL, SEQUENTIAL_BLUE, INK_SECONDARY

apply_style()

COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

with open(ROOT / "results" / "top_squares.json") as f:
    top_info = json.load(f)
TOP3 = top_info["top3_square_ids"]
FIXED_SQUARES = {4159: "Square 4159", 4556: "Square 4556"}
print("Top-3 squares (from 00_data_pipeline.ipynb):", TOP3)

In [ ]:
# Distribution of total traffic across all 10,000 squares.
totals = (
    pd.read_parquet(COMBINED_PATH, columns=["square_id", "internet_traffic"])
    .groupby("square_id")["internet_traffic"].sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(totals.values, bins=60, color=SEQUENTIAL_BLUE, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Total internet traffic per square, Nov 1 - Jan 1 (a.u.)")
ax.set_ylabel("Number of squares")
ax.set_title("Distribution of total internet traffic across 10,000 Milan squares")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_total_traffic_distribution.png")
plt.show()

print(totals.describe())
print(f"Skewness: {totals.skew():.3f}")
print(f"Zero-traffic squares: {(totals == 0).sum()}")

**Interpretation.** Total two-month traffic per square is heavily right-skewed:
the bulk of squares sit at low totals while a long tail of a few hundred
squares - presumably commercial/transit hubs - account for a disproportionate
share of traffic. The median square totals a small fraction of the busiest
square's total, and the interquartile range is narrow relative to that tail.
This skew is the reason we forecast each top-traffic square with its own
fitted model rather than pooling all 10,000 squares into one: a single global
model would be dominated by the low-traffic majority and would have little
incentive to fit the high-traffic squares' sharper peaks well.

In [ ]:
# Five-square comparison, first two weeks: top-3 by total traffic + two fixed
# reference squares (4159, 4556).
squares_to_plot = TOP3 + list(FIXED_SQUARES.keys())
labels = [f"Square {s} (rank {list(totals.index).index(s) + 1})" for s in TOP3] + list(FIXED_SQUARES.values())

first_two_weeks_end = pd.Timestamp("2013-11-01") + pd.Timedelta(days=14)

fig, axes = plt.subplots(len(squares_to_plot), 1, figsize=(9, 2.1 * len(squares_to_plot)), sharex=True)
for ax, sid, label, color in zip(axes, squares_to_plot, labels, CATEGORICAL):
    s = SquareSeries(sid, COMBINED_PATH).load()
    window = s.loc[:first_two_weeks_end]
    ax.plot(window.index, window.values, color=color, linewidth=1.2)
    ax.set_ylabel(label, fontsize=8, color=INK_SECONDARY, rotation=0, ha="right", va="center")
    ax.grid(True, alpha=0.5)
axes[-1].set_xlabel("Date (first two weeks: Nov 1 - Nov 14, 2013)")
fig.suptitle("Internet traffic, first two weeks, top-3 vs. fixed reference squares")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_top_areas_first_two_weeks.png")
plt.show()

**Interpretation.** All five squares show a clear daily cycle: overnight
troughs and one or two daytime peaks. The top-ranked square has the highest
and most variable peaks, including a visibly anomalous single-day spike well
above its typical daily maximum - a plausible event-driven surge rather than
routine demand. The rank-3 square's very first days look different from the
rest of its own series (flatter, lower) before settling into a regular
weekday/weekend cycle, showing that even a top-traffic square's pattern can
shift within the observation window. The two fixed reference squares (much
lower absolute traffic, consistent with their rank outside the top 3) still
show the same daily periodicity at smaller amplitude - the daily cycle is a
property of the network's usage pattern generally, not just of the busiest
cells.

In [ ]:
# Custom analysis 1: hour x day-of-week periodicity heatmap, top square.
top_square = TOP3[0]
s_top = SquareSeries(top_square, COMBINED_PATH).load()

hourly = s_top.to_frame("internet_traffic")
hourly["hour"] = hourly.index.hour
hourly["dow"] = hourly.index.dayofweek
pivot = hourly.pivot_table(index="hour", columns="dow", values="internet_traffic", aggfunc="mean")
pivot = pivot.reindex(columns=range(7))
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="Blues", origin="lower")
ax.set_xticks(range(7)); ax.set_xticklabels(day_labels)
ax.set_yticks(range(0, 24, 2)); ax.set_yticklabels(range(0, 24, 2))
ax.set_xlabel("Day of week"); ax.set_ylabel("Hour of day")
ax.set_title(f"Mean internet traffic by hour x weekday - square {top_square}")
fig.colorbar(im, ax=ax, label="Mean internet traffic (a.u.)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_periodicity_heatmap.png")
plt.show()

**Interpretation.** A single broad daytime peak (roughly midday through early
evening) and a deep overnight trough recur every day, but Saturday/Sunday
afternoons peak later and higher than weekday afternoons - a leisure-driven
rather than commute-driven usage pattern on weekends. This two-scale
periodicity (strong within-day, weaker but present day-of-week) is exactly
what motivates using both a Fourier daily-period term *and* an explicit
weekend indicator in the SARIMA and gradient-boosting input representations
in `02_experiments.ipynb`, rather than a single seasonal term.

In [ ]:
# Custom analysis 2: autocorrelation, partial autocorrelation, and stationarity.
fig, axes = plt.subplots(2, 1, figsize=(8, 6))
plot_acf(s_top, lags=576, ax=axes[0], color=SEQUENTIAL_BLUE)
axes[0].set_title(f"ACF (up to 4 days = 576 lags) - square {top_square}")
plot_pacf(s_top, lags=200, ax=axes[1], color=SEQUENTIAL_BLUE, method="ywm")
axes[1].set_title(f"PACF (up to 200 lags) - square {top_square}")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_acf_pacf.png")
plt.show()

adf_stat, adf_p, *_ = adfuller(s_top.values)
print(f"ADF statistic: {adf_stat:.4f}, p-value: {adf_p:.6f}")
print("Stationary at 5% level" if adf_p < 0.05 else "Non-stationary at 5% level")

**Interpretation.** The ACF shows a damped, periodic pattern with peaks
recurring every 144 lags (24 hours) - the signature of strong daily
seasonality - decaying only slowly across days. The PACF, in contrast, is
concentrated almost entirely in the first handful of lags (well under an
hour), with no meaningful *direct* dependence beyond that: the long-range
structure in the ACF is seasonal, mediated through the daily cycle, not
short-term momentum extending over hours. The Augmented Dickey-Fuller test
rejects the unit-root null decisively - the series is stationary in levels,
so no differencing is strictly required before modeling.

**Implication for forecasting**, tying directly into `02_experiments.ipynb`:
the PACF result justifies the gradient-boosting model's short-lag feature set
rather than a much longer lag window; the ACF's daily periodicity justifies
representing seasonality explicitly (Fourier terms for SARIMA, calendar
features for gradient boosting) rather than relying on differencing; and
stationarity in levels means the SARIMA order search doesn't need to assume
differencing away the trend, only test for it.